---
## Task 3 — Corner Kicks: Winning Teams vs. Losing Teams

### Analytic question formulation
Do **winning teams** earn significantly more **corner kicks per match** than **losing teams**?
(Corners are often used as a proxy for territorial/attacking dominance.)

### Data wrangling
Fetch the `TeamMatch` sheet from the clean dataset and keep only **decisive** results
(`Win`/`Loss` — draws are excluded since there is no "winner" to compare), dropping missing
`Corners` values.


In [2]:
import pandas as pd
import numpy as np
import re
from scipy import stats
CLEAN_PATH = "../World_Cup_2026_clean.xlsx"
team_match_t3 = pd.read_excel(CLEAN_PATH, sheet_name="TeamMatch")
decisive = team_match_t3[team_match_t3["Result"].isin(["Win", "Loss"])].dropna(subset=["Corners"])

print("Population size (decisive team-match records):", len(decisive))
print(decisive.groupby("Result")["Corners"].mean())

Population size (decisive team-match records): 158
Result
Loss    3.810127
Win     5.088608
Name: Corners, dtype: float64


### Data preparation and sampling
**Population:** 158 team-match records from decisive matches (79 wins, 79 losses).
**Sample:** an independent **simple random sample of n = 30** drawn separately from the winners
and from the losers, for a two-sample comparison.


In [3]:
win_sample  = decisive[decisive.Result == "Win"].sample(n=30, random_state=7)["Corners"]
loss_sample = decisive[decisive.Result == "Loss"].sample(n=30, random_state=7)["Corners"]
print("Winners sample n:", len(win_sample), " Losers sample n:", len(loss_sample))

Winners sample n: 30  Losers sample n: 30


### Descriptive statistics

In [4]:
for name, s in [("Winning teams", win_sample), ("Losing teams", loss_sample)]:
    print(f"\n{name}:")
    print(s.describe())


Winning teams:
count    30.000000
mean      4.933333
std       3.876840
min       0.000000
25%       2.000000
50%       4.500000
75%       7.000000
max      19.000000
Name: Corners, dtype: float64

Losing teams:
count    30.000000
mean      3.766667
std       2.269488
min       1.000000
25%       2.000000
50%       3.500000
75%       5.000000
max      11.000000
Name: Corners, dtype: float64


### Inferential statistics — Confidence intervals (95%, each group)

In [5]:
for name, s in [("Winning teams", win_sample), ("Losing teams", loss_sample)]:
    m, sem = s.mean(), stats.sem(s)
    ci = stats.t.interval(0.95, df=len(s) - 1, loc=m, scale=sem)
    print(f"{name}: mean = {m:.3f}, 95% CI = ({ci[0]:.3f}, {ci[1]:.3f})")

Winning teams: mean = 4.933, 95% CI = (3.486, 6.381)
Losing teams: mean = 3.767, 95% CI = (2.919, 4.614)


### Inferential statistics — Two-sample t-Test (Welch's)
H₀: μ(win) = μ(loss)  vs.  H₁: μ(win) ≠ μ(loss)


In [6]:
t_stat, p_val = stats.ttest_ind(win_sample, loss_sample, equal_var=False)
print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.4f}")
alpha = 0.05
print("Conclusion:", "Reject H0" if p_val < alpha else "Fail to reject H0",
      "at the 5% significance level.")

t-statistic = 1.422, p-value = 0.1615
Conclusion: Fail to reject H0 at the 5% significance level.


**Interpretation:** Winning teams averaged noticeably more corners in the sample (4.93 vs
3.77), but with p = 0.162 (> 0.05) this sample does not provide statistically significant
evidence at the 5% level, and the CIs overlap. A larger sample might narrow this.
